# Phase 3 -- Transformer Fine-Tuning (DeBERTa-v3-base)
## Human vs. AI-Generated Text Classification

**Memory-Efficient Training Configuration:**

| Setting | Value | Reason |
|---------|-------|--------|
| | 512 | Keeps full text |
| | 2 | Avoids OOM with 512-token inputs |
| | 8 | Effective batch = 16 with only 2 in memory |
| | 0 | avoid pickling issues with notebook-defined classes via |
| | 4 | limits thread contention for responsiveness |

**Architecture:** DeBERTa-v3-base + Focal Loss (gamma = 2.0) + class weights 
**Scheduler:** Cosine with 10% warmup, LLRD (factor = 0.9) 
**Data split:** Train (70%) / Val (15%) / Cal (15%) 
**Early stopping:** Patience = 2 epochs


In [ ]:
import os, warnings, gc, time
os.chdir('/Users/aliivaezii/Documents/MALTO')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ==================== CONFIG ====================
SEED = 42
NUM_LABELS = 6
MODEL_NAME = 'microsoft/deberta-v3-base'

# ---------- Memory-safe training settings ----------
# MAX_LEN=512 gave 0.9071 F1 but OOM'd at BATCH_SIZE=8.
# MAX_LEN=256 only gave 0.5063 F1 (61% texts truncated!).
# Solution: Keep MAX_LEN=512 but use BATCH_SIZE=2 + GRAD_ACCUM=8 → eff=16
MAX_LEN = 512
BATCH_SIZE = 2
GRAD_ACCUM = 8 # effective batch = 2*8 = 16

EPOCHS = 5
LR = 3e-5 # slightly higher than 2e-5, works well with warmup
WEIGHT_DECAY = 0.01
PATIENCE = 2 # early stopping: stop if no improvement for 2 epochs
NUM_WORKERS = 0 # avoid pickling issues with notebook-defined classes
LABEL_MAP = {0:'Human', 1:'DeepSeek', 2:'Grok', 3:'Claude', 4:'Gemini', 5:'ChatGPT'}

# Limit torch threads to stay responsive during training
torch.set_num_threads(4)

np.random.seed(SEED)
torch.manual_seed(SEED)

# Free any leftover memory from previous runs
gc.collect()
DEVICE = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
if DEVICE.type == 'mps':
    torch.mps.empty_cache()

print(f'Device: {DEVICE}')
print(f'Torch threads: {torch.get_num_threads()}')

# Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'MAX_LEN={MAX_LEN}, BATCH={BATCH_SIZE}×{GRAD_ACCUM}=eff {BATCH_SIZE*GRAD_ACCUM}, LR={LR}, EPOCHS={EPOCHS}')

### Overfitting and Underfitting Prevention

Transformer fine-tuning is prone to overfitting on small datasets (2,400 samples). The following strategies are applied:

- **Early stopping (patience=2)**: Training halts if validation F1 does not improve for 2 consecutive epochs, preventing the model from memorizing the training set.
- **Weight decay (0.01)**: L2 regularization on all parameters except biases and LayerNorm weights.
- **Layer-wise learning rate decay (LLRD, factor=0.9)**: Lower transformer layers receive smaller learning rates, preserving pre-trained representations and reducing catastrophic forgetting.
- **Gradient clipping (max_norm=1.0)**: Prevents exploding gradients that can destabilize training.
- **Focal Loss (gamma=2.0)**: Focuses training on hard-to-classify examples rather than over-optimizing easy ones, which improves generalization on minority classes.
- **Cosine learning rate schedule with warmup**: Gradually reduces the learning rate, preventing sharp parameter updates in later epochs.
- **Separate calibration set**: Temperature scaling is performed on a held-out calibration set (15% of data), ensuring probability calibration does not overfit to the validation set.


## 1. Pre-tokenize ALL data upfront
This is the key speedup -- tokenize once, reuse every epoch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer: {MODEL_NAME}, vocab={tokenizer.vocab_size}')

t0 = time.time()

# Pre-tokenize train
print('Pre-tokenizing train...')
train_encodings = tokenizer(
    train_df['TEXT'].tolist(),
    truncation=True, padding='max_length',
    max_length=MAX_LEN, return_tensors='pt'
    )
print(f' input_ids: {train_encodings["input_ids"].shape}')

# Pre-tokenize test
print('Pre-tokenizing test...')
test_encodings = tokenizer(
    test_df['TEXT'].tolist(),
    truncation=True, padding='max_length',
    max_length=MAX_LEN, return_tensors='pt'
    )
print(f' input_ids: {test_encodings["input_ids"].shape}')

print(f'\nPre-tokenization done in {time.time()-t0:.1f}s')

# Check truncation stats
n_truncated = (train_encodings['attention_mask'].sum(dim=1) == MAX_LEN).sum().item()
print(f'Train samples truncated at {MAX_LEN}: {n_truncated}/{len(train_df)} ({n_truncated/len(train_df)*100:.1f}%)')

### 📦 Dataset Class

A custom PyTorch `Dataset` that wraps pre-tokenized tensors for efficient batching — no redundant tokenization per epoch.

In [ ]:
class PreTokenizedDataset(Dataset):
    """Fast dataset -- data is already tokenized, just index into tensors."""
    def __init__(self, encodings, labels=None, indices=None):
        self.encodings = encodings
        self.labels = labels
        self.indices = indices # subset indices (for train/val splits)
 
        def __len__(self):
            return len(self.indices) if self.indices is not None else self.encodings['input_ids'].shape[0]
 
            def __getitem__(self, idx):
                real_idx = self.indices[idx] if self.indices is not None else idx
                item = {
                    'input_ids': self.encodings['input_ids'][real_idx],
                    'attention_mask': self.encodings['attention_mask'][real_idx],
                    }
                if 'token_type_ids' in self.encodings:
                    item['token_type_ids'] = self.encodings['token_type_ids'][real_idx]
                    if self.labels is not None:
                        item['labels'] = torch.tensor(self.labels[real_idx], dtype=torch.long)
                        return item

print('PreTokenizedDataset defined -- zero tokenization overhead per batch!')

## 2. Train / Validation / Calibration Split (Professor's Method)
**4-part approach**: Train (70%) / Val (15%) / Calibration (15%) / Test (Kaggle)

- **Train**: model learns weights
- **Validation**: early stopping / best epoch selection
- **Calibration**: temperature scaling for well-calibrated probabilities
- **Test**: Kaggle test set (no labels)

In [ ]:
y_all = train_df['LABEL'].values

# First split: 70% train, 30% holdout
train_idx, holdout_idx = train_test_split(
    np.arange(len(y_all)), test_size=0.30,
    stratify=y_all, random_state=SEED
    )

# Second split: holdout → 50/50 → val (15%) + calibration (15%)
val_idx, cal_idx = train_test_split(
    holdout_idx, test_size=0.50,
    stratify=y_all[holdout_idx], random_state=SEED
    )

print(f'Train: {len(train_idx):4d} samples ({len(train_idx)/len(y_all)*100:.0f}%)')
print(f'Validation: {len(val_idx):4d} samples ({len(val_idx)/len(y_all)*100:.0f}%)')
print(f'Calibration: {len(cal_idx):4d} samples ({len(cal_idx)/len(y_all)*100:.0f}%)')
print(f'Test: {len(test_df):4d} samples (Kaggle)')

# Verify stratification
print('\nLabel distribution per split:')
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Cal', cal_idx)]:
    counts = pd.Series(y_all[idx]).value_counts().sort_index()
    pcts = (counts / counts.sum() * 100).round(1)
    print(f' {name:5s}: ' + ' '.join(f'{LABEL_MAP[i][:5]}={counts[i]}({pcts[i]}%)' for i in range(6)))

## 3. Focal Loss + Class Weights

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss -- focuses on hard examples, great for imbalanced data."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
 
        def forward(self, logits, targets):
            ce = nn.functional.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
            pt = torch.exp(-ce)
            return (((1 - pt) ** self.gamma) * ce).mean()

            # Class weights from TRAINING set only (not val/cal)
train_counts = pd.Series(y_all[train_idx]).value_counts().sort_index().values
cw = 1.0 / train_counts
cw = cw / cw.sum() * NUM_LABELS
CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

print('Class weights (from train split):')
for i in range(NUM_LABELS):
    print(f' {LABEL_MAP[i]:10s}: n={train_counts[i]:4d}, w={cw[i]:.3f}')

## 4. Training Loop (with gradient accumulation)

In [ ]:
def get_optimizer_with_llrd(model, lr=LR, weight_decay=WEIGHT_DECAY, llrd_factor=0.9):
    """Layer-wise Learning Rate Decay -- lower layers get smaller LR.
    This is a proven technique for fine-tuning DeBERTa (+1-3% F1)."""
    opt_params = []
    no_decay = ['bias', 'LayerNorm.weight', 'layernorm.weight']
 
    # Classifier head gets full LR
    head_params_wd = [p for n, p in model.named_parameters() 
        if ('classifier' in n or 'pooler' in n) and not any(nd in n for nd in no_decay)]
    head_params_nowd = [p for n, p in model.named_parameters() 
        if ('classifier' in n or 'pooler' in n) and any(nd in n for nd in no_decay)]
    opt_params.append({'params': head_params_wd, 'lr': lr, 'weight_decay': weight_decay})
    opt_params.append({'params': head_params_nowd, 'lr': lr, 'weight_decay': 0.0})
 
    # Encoder layers -- deeper layers get lower LR
    num_layers = model.config.num_hidden_layers # 12 for base
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd_factor ** (num_layers - 1 - layer_idx))
        layer_params_wd = [p for n, p in model.named_parameters()
            if f'encoder.layer.{layer_idx}.' in n and not any(nd in n for nd in no_decay)]
        layer_params_nowd = [p for n, p in model.named_parameters()
            if f'encoder.layer.{layer_idx}.' in n and any(nd in n for nd in no_decay)]
        if layer_params_wd:
            opt_params.append({'params': layer_params_wd, 'lr': layer_lr, 'weight_decay': weight_decay})
            if layer_params_nowd:
                opt_params.append({'params': layer_params_nowd, 'lr': layer_lr, 'weight_decay': 0.0})
 
                # Embeddings get the smallest LR
                emb_lr = lr * (llrd_factor ** num_layers)
                emb_params_wd = [p for n, p in model.named_parameters()
                    if 'embeddings' in n and not any(nd in n for nd in no_decay)]
                emb_params_nowd = [p for n, p in model.named_parameters()
                    if 'embeddings' in n and any(nd in n for nd in no_decay)]
                if emb_params_wd:
                    opt_params.append({'params': emb_params_wd, 'lr': emb_lr, 'weight_decay': weight_decay})
                    if emb_params_nowd:
                        opt_params.append({'params': emb_params_nowd, 'lr': emb_lr, 'weight_decay': 0.0})
 
                        # Catch any remaining params (e.g. encoder.LayerNorm) -- give them mid-range LR
                        assigned = set()
                        for group in opt_params:
                            for p in group['params']:
                                assigned.add(id(p))
                                remaining_wd = [p for n, p in model.named_parameters() 
                                    if id(p) not in assigned and not any(nd in n for nd in no_decay)]
                                remaining_nowd = [p for n, p in model.named_parameters()
                                    if id(p) not in assigned and any(nd in n for nd in no_decay)]
                                if remaining_wd:
                                    opt_params.append({'params': remaining_wd, 'lr': lr * 0.5, 'weight_decay': weight_decay})
                                    if remaining_nowd:
                                        opt_params.append({'params': remaining_nowd, 'lr': lr * 0.5, 'weight_decay': 0.0})
 
                                        total_opt = sum(len(g['params']) for g in opt_params)
                                        total_model = sum(1 for _ in model.parameters())
                                        assert total_opt == total_model, f'LLRD mismatch: {total_opt} vs {total_model} params!'
 
                                        return torch.optim.AdamW(opt_params)


def train_model(train_idx, val_idx, cal_idx, epochs=EPOCHS):
    """Train DeBERTa with LLRD, gradient accumulation, and early stopping."""
 
    # DataLoaders from pre-tokenized data
    train_ds = PreTokenizedDataset(train_encodings, y_all, train_idx)
    val_ds = PreTokenizedDataset(train_encodings, y_all, val_idx)
    test_ds = PreTokenizedDataset(test_encodings)
 
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2,
        num_workers=NUM_WORKERS, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2,
        num_workers=NUM_WORKERS, pin_memory=False)
 
    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
        )
    model.to(DEVICE)
 
    criterion = FocalLoss(alpha=CLASS_WEIGHTS, gamma=2.0)
 
    # LLRD optimizer -- proven technique for DeBERTa fine-tuning
    optimizer = get_optimizer_with_llrd(model, lr=LR, llrd_factor=0.9)
 
    total_steps = (len(train_loader) // GRAD_ACCUM) * epochs
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)
 
    best_f1 = 0
    best_state = None
    patience_counter = 0
    history = []
 
    for epoch in range(epochs):
        t0 = time.time()
 
        # --- TRAIN ---
        model.train()
        total_loss = 0
        optimizer.zero_grad()
 
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
            desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False,
            bar_format='{l_bar}{bar:30}{r_bar}')
 
        for step, batch in pbar:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
 
            outputs = model(**inputs)
            loss = criterion(outputs.logits, labels) / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
 
            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
 
                # Update progress bar with running loss
                running_loss = total_loss / (step + 1)
                pbar.set_postfix({'loss': f'{running_loss:.4f}', 
                    'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
 
                pbar.close()
 
                # Flush remaining gradients
                if (step + 1) % GRAD_ACCUM != 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
 
                    avg_loss = total_loss / len(train_loader)
 
                    # --- VALIDATE ---
                    model.eval()
                    all_preds, all_labels = [], []
                    with torch.no_grad():
                        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]',
                            leave=False, bar_format='{l_bar}{bar:30}{r_bar}'):
                                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                                labels = inputs.pop('labels')
                                outputs = model(**inputs)
                                all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                                all_labels.extend(labels.cpu().numpy())
 
                                val_f1 = f1_score(all_labels, all_preds, average='macro')
                                elapsed = time.time() - t0
                                history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_f1': val_f1})
 
                                if val_f1 > best_f1:
                                    best_f1 = val_f1
                                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                                    patience_counter = 0
                                    marker = ' new best!'
                                else:
                                    patience_counter += 1
                                    marker = f' (patience {patience_counter}/{PATIENCE})'
 
                                    print(f' Epoch {epoch+1}/{epochs} | loss={avg_loss:.4f} | val_f1={val_f1:.4f} | {elapsed:.0f}s{marker}')
 
                                    if patience_counter >= PATIENCE:
                                        print(f' Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
                                        break
 
                                        # Load best weights & generate predictions
                                        model.load_state_dict(best_state)
                                        model.eval()
 
                                        # Val predictions (for evaluation metrics)
                                        val_preds, val_logits_list = [], []
                                        with torch.no_grad():
                                            for batch in tqdm(val_loader, desc='Predicting [Val]', leave=False,
                                                bar_format='{l_bar}{bar:30}{r_bar}'):
                                                    inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                                                    inputs.pop('labels', None)
                                                    outputs = model(**inputs)
                                                    val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                                                    val_logits_list.extend(outputs.logits.cpu().numpy())
 
                                                    # Test predictions
                                                    test_logits_list = []
                                                    with torch.no_grad():
                                                        for batch in tqdm(test_loader, desc='Predicting [Test]', leave=False,
                                                            bar_format='{l_bar}{bar:30}{r_bar}'):
                                                                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                                                                outputs = model(**inputs)
                                                                test_logits_list.extend(outputs.logits.cpu().numpy())
 
                                                                # Calibration set logits (unseen during training!)
                                                                cal_ds = PreTokenizedDataset(train_encodings, y_all, cal_idx)
                                                                cal_loader = DataLoader(cal_ds, batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS)
                                                                cal_logits_list, cal_labels_list = [], []
                                                                with torch.no_grad():
                                                                    for batch in tqdm(cal_loader, desc='Predicting [Cal]', leave=False,
                                                                        bar_format='{l_bar}{bar:30}{r_bar}'):
                                                                            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                                                                            labels = inputs.pop('labels')
                                                                            outputs = model(**inputs)
                                                                            cal_logits_list.extend(outputs.logits.cpu().numpy())
                                                                            cal_labels_list.extend(labels.cpu().numpy())
 
                                                                            print(f' Best val F1: {best_f1:.4f}')
 
                                                                            # Save best model weights before cleanup
                                                                            os.makedirs('models', exist_ok=True)
                                                                            os.makedirs('checkpoints', exist_ok=True)
                                                                            torch.save(model.state_dict(), 'models/deberta_v3_base_best.pt')
                                                                            torch.save(model.state_dict(), 'checkpoints/deberta_v3_base_best.pt')
                                                                            print(f' Model checkpoint saved to checkpoints/deberta_v3_base_best.pt')
                                                                            print(f' Model weights saved to models/deberta_v3_base_best.pt')
 
                                                                            # Cleanup
                                                                            del model, optimizer, scheduler, best_state
                                                                            gc.collect()
                                                                            if DEVICE.type == 'mps':
                                                                                torch.mps.empty_cache()
 
                                                                                return {
                                                                                    'best_f1': best_f1,
                                                                                    'val_preds': np.array(val_preds),
                                                                                    'val_logits': np.array(val_logits_list),
                                                                                    'test_logits': np.array(test_logits_list),
                                                                                    'cal_logits': np.array(cal_logits_list),
                                                                                    'cal_labels': np.array(cal_labels_list),
                                                                                    'history': history,
                                                                                    }

print('Training function ready.')
print(' LLRD (layer-wise LR decay, factor=0.9)')
print(' Cosine scheduler with 10% warmup')
print(' Gradient accumulation (eff batch=16)')
print(f' Early stopping (patience={PATIENCE})')
print(f' Focal Loss (gamma=2.0) + class weights')
print(' tqdm progress bars on all loops')

## 5. Train the Model

In [ ]:
print(f'Training DeBERTa-v3-base')
print(f' Train: {len(train_idx)}, Val: {len(val_idx)}, Cal: {len(cal_idx)}')
print(f' MAX_LEN: {MAX_LEN}')
print(f' Batch: {BATCH_SIZE} × accum {GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}')
print(f' Epochs: {EPOCHS} (early stop patience={PATIENCE})')
print(f' LR: {LR} with LLRD factor=0.9 + cosine schedule')
print(f' Workers: {NUM_WORKERS}, Threads: {torch.get_num_threads()}')
print()

t_start = time.time()
result = train_model(train_idx, val_idx, cal_idx, epochs=EPOCHS)
t_total = time.time() - t_start

print(f'\nTotal training time: {t_total/60:.1f} minutes')

## 6. Temperature Scaling (Calibration)
Use the held-out calibration set to find optimal temperature for probability calibration.

In [ ]:
class TemperatureScaler:
    """Post-hoc temperature scaling for calibrated probabilities."""
    def __init__(self):
        self.temperature = 1.0
 
        def fit(self, logits, labels):
            """Find optimal temperature on calibration set."""
            logits_t = torch.tensor(logits, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.long)
 
            best_temp = 1.0
            best_nll = float('inf')
 
            for temp in np.arange(0.5, 5.0, 0.05):
                scaled = logits_t / temp
                nll = nn.functional.cross_entropy(scaled, labels_t).item()
                if nll < best_nll:
                    best_nll = nll
                    best_temp = temp
 
                    self.temperature = best_temp
 
                    # Compare F1 before/after scaling
                    preds_before = logits_t.argmax(-1).numpy()
                    preds_after = (logits_t / self.temperature).argmax(-1).numpy()
                    f1_before = f1_score(labels, preds_before, average='macro')
                    f1_after = f1_score(labels, preds_after, average='macro')
 
                    print(f' Optimal temperature: {self.temperature:.2f}')
                    print(f' Cal set NLL: {best_nll:.4f}')
                    print(f' Cal set F1: before={f1_before:.4f}, after={f1_after:.4f}')
                    return self
 
                    def predict_proba(self, logits):
                        """Get calibrated probabilities."""
                        scaled = torch.tensor(logits, dtype=torch.float32) / self.temperature
                        return torch.softmax(scaled, dim=-1).numpy()

                        # Fit temperature scaler on calibration set
print('Calibrating probabilities on held-out calibration set...')
scaler = TemperatureScaler()
scaler.fit(result['cal_logits'], result['cal_labels'])

# Get calibrated test probabilities
test_probs_calibrated = scaler.predict_proba(result['test_logits'])
test_preds_calibrated = test_probs_calibrated.argmax(axis=1)

# Also get uncalibrated for comparison
test_probs_raw = torch.softmax(torch.tensor(result['test_logits']), dim=-1).numpy()
test_preds_raw = result['test_logits'].argmax(axis=1)

print(f'\nRaw test preds distribution:')
print(pd.Series(test_preds_raw).value_counts().sort_index().to_string())
print(f'\nCalibrated test preds distribution:')
print(pd.Series(test_preds_calibrated).value_counts().sort_index().to_string())

## 7. Evaluation

In [ ]:
# Validation set results
val_labels = y_all[val_idx]
val_preds = result['val_preds']
val_f1 = f1_score(val_labels, val_preds, average='macro')

print(f'Validation Macro F1: {val_f1:.4f}')
print('\nPer-class report:')
print(classification_report(val_labels, val_preds,
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))

# Calibration set results (unseen during training)
cal_labels = result['cal_labels']
cal_preds = result['cal_logits'].argmax(axis=1)
cal_f1 = f1_score(cal_labels, cal_preds, average='macro')
print(f'Calibration set Macro F1: {cal_f1:.4f}')

# Combined holdout (val + cal)
all_holdout_idx = np.concatenate([val_idx, cal_idx])
all_holdout_labels = y_all[all_holdout_idx]
all_holdout_preds = np.concatenate([val_preds, cal_preds])
holdout_f1 = f1_score(all_holdout_labels, all_holdout_preds, average='macro')
print(f'Combined holdout F1 (val+cal): {holdout_f1:.4f}')

### 🔍 Confusion Matrix

Visualise the validation confusion matrix to identify per-class strengths and weaknesses.

In [ ]:
# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
    xticklabels=[LABEL_MAP[i] for i in range(6)],
    yticklabels=[LABEL_MAP[i] for i in range(6)])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'DeBERTa-v3-base + Focal Loss + Calibration\nVal Macro F1 = {val_f1:.4f}')
plt.tight_layout()
plt.savefig('figures/deberta_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion matrix.')

### ✅ Validation Results

Evaluate the fine-tuned model on the held-out validation split.

In [ ]:
# Compare with Phase 2
print('='*60)
print('MODEL COMPARISON')
print('='*60)
print(f' Phase 2 SVC (C=5.0): 5-fold CV F1 = 0.9194')
print(f' Phase 2 Ensemble (Top-5): OOF F1 = 0.9176')
print(f' Phase 3 DeBERTa (val): Val F1 = {val_f1:.4f}')
print(f' Phase 3 DeBERTa (cal): Cal F1 = {cal_f1:.4f}')
print(f' Phase 3 DeBERTa (holdout): Holdout F1 = {holdout_f1:.4f}')

print(f'\n Temperature: {scaler.temperature:.2f}')
print(f' Training time: {t_total/60:.1f} min')

## 8. Save Predictions

### 🔍 Confusion Matrix Analysis

Visualise per-class prediction patterns to identify systematic misclassifications.

In [ ]:
# Use calibrated predictions
preds = test_preds_calibrated

# Ensure output dirs exist
os.makedirs('submissions', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Save submission (ID,LABEL format)
sub_path = 'submissions/transformer_deberta_calibrated.csv'
with open(sub_path, 'w') as f:
    lines = ['ID,LABEL'] + [f'{i},{preds[i]}' for i in range(len(preds))]
    f.write('\n'.join(lines) + '\n')
print(f'Submission saved: {sub_path}')

# Save logits & probabilities for ensembling (Phase 4+)
np.save('models/transformer_test_logits.npy', result['test_logits'])
np.save('models/transformer_test_probs_calibrated.npy', test_probs_calibrated)
np.save('models/transformer_test_probs_raw.npy', test_probs_raw)
np.save('models/transformer_val_preds.npy', result['val_preds'])
np.save('models/transformer_val_logits.npy', result['val_logits'])
np.save('models/transformer_cal_logits.npy', result['cal_logits'])

# Save split indices for reproducibility
np.savez('models/transformer_split_indices.npz',
    train_idx=train_idx, val_idx=val_idx, cal_idx=cal_idx)


# Save temperature for reproducibility
import json
transformer_config = {
    'model_name': MODEL_NAME,
    'temperature': float(scaler.temperature),
    'val_f1': float(val_f1),
    'cal_f1': float(cal_f1),
    'holdout_f1': float(holdout_f1),
    'max_len': MAX_LEN,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'lr': LR,
    'epochs': EPOCHS,
    }
with open('models/transformer_config.json', 'w') as f:
    json.dump(transformer_config, f, indent=2)
print('Saved: models/transformer_config.json')
print(f'Saved: logits {result["test_logits"].shape}, probs {test_probs_calibrated.shape}')
print(' Phase 3 predictions ready!')

### 🏁 Phase 3 — Summary

Final summary of DeBERTa fine-tuning results, including validation F1, calibration, and saved artifacts.

In [ ]:
# --- Final Summary ---
print('='*60)
print('PHASE 3 SUMMARY')
print('='*60)
print(f'\nModel: {MODEL_NAME}')
print(f'Loss: Focal Loss (gamma=2.0) + class weights')
print(f'Split: Train({len(train_idx)})/Val({len(val_idx)})/Cal({len(cal_idx)})/Test({len(test_df)})')
print(f'Max len: {MAX_LEN}')
print(f'Eff. batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'Epochs: {EPOCHS}')
print(f'LR: {LR}')
print(f'Temperature: {scaler.temperature:.2f}')
print(f'\nVal F1: {val_f1:.4f}')
print(f'Cal F1: {cal_f1:.4f}')
print(f'Holdout F1: {holdout_f1:.4f}')
print(f'Training time: {t_total/60:.1f} min')
print(f'\nPrediction distribution:')
for label, count in pd.Series(preds).value_counts().sort_index().items():
    print(f' {LABEL_MAP[label]:10s}: {count}')

### 🏁 Phase 3 — Final Summary

Summary of DeBERTa fine-tuning results, calibration, and saved artifacts.

### 📤 Calibrated Test Predictions

Apply temperature scaling to the test logits and generate the final calibrated predictions.